## Data Loading & Prep

In [3]:
import pandas as pd
import numpy as np

# Function to optimize dataframe memory usage
def optimize_dataframe(df):
    # Convert numeric columns to more memory-efficient types
    for col in df.select_dtypes(include=['int64', 'float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer' if 'int' in str(df[col].dtype) else 'float')
    return df

# Load and optimize datasets (do this before converting to Dask)
calendar = pd.read_csv('calendar.csv')
sales = pd.read_csv('sales_train_validation.csv')
prices = pd.read_csv('sell_prices.csv')

In [4]:
# Convert the relevant columns to categories in pandas before converting to Dask
calendar['event_name_1'] = calendar['event_name_1'].astype('category')
calendar['event_type_1'] = calendar['event_type_1'].astype('category')
calendar['weekday'] = calendar['weekday'].astype('category')
calendar['snap_CA'] = calendar['snap_CA'].astype(np.int8)
calendar['snap_TX'] = calendar['snap_TX'].astype(np.int8)
calendar['snap_WI'] = calendar['snap_WI'].astype(np.int8)

# Convert prices columns to category type
prices['store_id'] = prices['store_id'].astype('category')
prices['item_id'] = prices['item_id'].astype('category')

# Convert sales columns to category type
sales['item_id'] = sales['item_id'].astype('category')
sales['dept_id'] = sales['dept_id'].astype('category')
sales['cat_id'] = sales['cat_id'].astype('category')
sales['store_id'] = sales['store_id'].astype('category')
sales['state_id'] = sales['state_id'].astype('category')


In [4]:
# Check for missing values
print("Missing values in calendar data:\n", calendar.isnull().sum())
print("Missing values in sales data:\n", sales.isnull().sum())
print("Missing values in prices data:\n", prices.isnull().sum())

Missing values in calendar data:
 date               0
wm_yr_wk           0
weekday            0
wday               0
month              0
year               0
d                  0
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
snap_CA            0
snap_TX            0
snap_WI            0
dtype: int64
Missing values in sales data:
 id          0
item_id     0
dept_id     0
cat_id      0
store_id    0
           ..
d_1909      0
d_1910      0
d_1911      0
d_1912      0
d_1913      0
Length: 1919, dtype: int64
Missing values in prices data:
 store_id      0
item_id       0
wm_yr_wk      0
sell_price    0
dtype: int64


In [9]:
# we decided to drop event 2 due to huge amount of null value
print("Unique values for event_name_1:\n", calendar['event_name_1'].unique())
print("Unique values for event_type_1:\n", calendar['event_type_1'].unique())
print("Unique values for event_name_2:\n", calendar['event_name_2'].unique())
print("Unique values for event_type_2:\n", calendar['event_type_2'].unique())

Unique values for event_name_1:
 [nan 'SuperBowl' 'ValentinesDay' 'PresidentsDay' 'LentStart' 'LentWeek2'
 'StPatricksDay' 'Purim End' 'OrthodoxEaster' 'Pesach End' 'Cinco De Mayo'
 "Mother's day" 'MemorialDay' 'NBAFinalsStart' 'NBAFinalsEnd'
 "Father's day" 'IndependenceDay' 'Ramadan starts' 'Eid al-Fitr'
 'LaborDay' 'ColumbusDay' 'Halloween' 'EidAlAdha' 'VeteransDay'
 'Thanksgiving' 'Christmas' 'Chanukah End' 'NewYear' 'OrthodoxChristmas'
 'MartinLutherKingDay' 'Easter']
Unique values for event_type_1:
 [nan 'Sporting' 'Cultural' 'National' 'Religious']
Unique values for event_name_2:
 [nan 'Easter' 'Cinco De Mayo' 'OrthodoxEaster' "Father's day"]
Unique values for event_type_2:
 [nan 'Cultural' 'Religious']


In [5]:
import dask.dataframe as dd
calendar = dd.from_pandas(calendar, npartitions=4)
sales = dd.from_pandas(sales, npartitions=4)
prices = dd.from_pandas(prices, npartitions=4)

# Add "No Event" as a category before filling NaNs (ensure the column is a category first)
calendar['event_name_1'] = calendar['event_name_1'].cat.add_categories("No Event")
calendar['event_type_1'] = calendar['event_type_1'].cat.add_categories("No Event")

# Fill NaN values with "No Event"
calendar['event_name_1'] = calendar['event_name_1'].fillna("No Event")
calendar['event_type_1'] = calendar['event_type_1'].fillna("No Event")

# Drop event 2 columns
calendar = calendar.drop(columns=['event_name_2', 'event_type_2'])

# Melt sales data
sales_melted = sales.melt(id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
                           var_name='d', value_name='sales')

# Merge with calendar data
sales_melted = sales_melted.merge(calendar.compute(), how='left', on='d')

# Merge with prices data
sales_melted = sales_melted.merge(prices.compute(), how='left', on=['store_id', 'item_id', 'wm_yr_wk'])

# Save intermediate data as Parquet (optional step)
sales_melted.to_parquet('sales_melted.parquet', engine='pyarrow', compression='snappy')

/usr/local/lib/python3.11/dist-packages/dask/dataframe/multi.py:521: UserWarning: Merging dataframes with merge column data type mismatches: 
+---------------+------------+-------------+
| Merge columns | left dtype | right dtype |
+---------------+------------+-------------+
| ('d', 'd')    | object     | string      |
+---------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/dask/dataframe/multi.py:521: UserWarning: Merging dataframes with merge column data type mismatches: 
+--------------------------+------------+-------------+
| Merge columns            | left dtype | right dtype |
+--------------------------+------------+-------------+
| ('wm_yr_wk', 'wm_yr_wk') | float64    | int64       |
+--------------------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(


## Feature Engineering

In [10]:
import dask.dataframe as dd
import pandas as pd

# Function for feature engineering on Dask DataFrame partitions
def create_time_series_features_dask(df):
    # Convert 'date' to datetime, handling errors with 'coerce'
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # Ensure the date conversion worked (you can add more validation if needed)
    if df['date'].isnull().sum() > 0:
        print(f"Warning: There are {df['date'].isnull().sum()} invalid date entries.")

    # Create time-based features
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['day_of_week'] = df['date'].dt.weekday

    # Lag features
    df['lag_1'] = df.groupby(['item_id', 'store_id'])['sales'].shift(1)
    df['lag_7'] = df.groupby(['item_id', 'store_id'])['sales'].shift(7)
    df['lag_30'] = df.groupby(['item_id', 'store_id'])['sales'].shift(30)

    # Rolling mean features
    df['rolling_mean_7'] = df.groupby(['item_id', 'store_id'])['sales'].rolling(7).mean().reset_index(level=[0, 1], drop=True)
    df['rolling_mean_30'] = df.groupby(['item_id', 'store_id'])['sales'].rolling(30).mean().reset_index(level=[0, 1], drop=True)

    # Fill NaN values with 0
    df.fillna(0, inplace=True)
    return df

# Create a dummy meta DataFrame to specify the output structure (needed for Dask)
meta = pd.DataFrame({
    'id': pd.Series([], dtype='int64'),
    'item_id': pd.Series([], dtype='category'),
    'dept_id': pd.Series([], dtype='category'),
    'cat_id': pd.Series([], dtype='category'),
    'store_id': pd.Series([], dtype='category'),
    'state_id': pd.Series([], dtype='category'),
    'date': pd.Series([], dtype='datetime64[ns]'),
    'sales': pd.Series([], dtype='float64'),
    'day': pd.Series([], dtype='int64'),
    'month': pd.Series([], dtype='int64'),
    'year': pd.Series([], dtype='int64'),
    'day_of_week': pd.Series([], dtype='int64'),
    'lag_1': pd.Series([], dtype='float64'),
    'lag_7': pd.Series([], dtype='float64'),
    'lag_30': pd.Series([], dtype='float64'),
    'rolling_mean_7': pd.Series([], dtype='float64'),
    'rolling_mean_30': pd.Series([], dtype='float64')
})

# Apply the function to the Dask DataFrame with the meta argument
sales_melted = sales_melted.map_partitions(create_time_series_features_dask, meta=meta)

# Optional: If you want to view the result in Pandas
# sales_melted = sales_melted.compute()

# Check the Dask DataFrame status after operation
print(sales_melted.head())


Dask DataFrame Structure:
                  id          item_id          dept_id           cat_id         store_id         state_id            date    sales    day  month   year day_of_week    lag_1    lag_7   lag_30 rolling_mean_7 rolling_mean_30
npartitions=4                                                                                                                                                                                                                
               int64  category[known]  category[known]  category[known]  category[known]  category[known]  datetime64[ns]  float64  int64  int64  int64       int64  float64  float64  float64        float64         float64
                 ...              ...              ...              ...              ...              ...             ...      ...    ...    ...    ...         ...      ...      ...      ...            ...             ...
                 ...              ...              ...              ...              .

In [ ]:
import dask.dataframe as dd
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Function for feature engineering on Dask DataFrame partitions
def create_time_series_features_dask(df):
    # Convert 'date' to datetime, handling errors with 'coerce'
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # Ensure the date conversion worked
    if df['date'].isnull().sum() > 0:
        print(f"Warning: There are {df['date'].isnull().sum()} invalid date entries.")

    # Create time-based features
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['day_of_week'] = df['date'].dt.weekday

    # Lag features
    df['lag_1'] = df.groupby(['item_id', 'store_id'], observed=False)['sales'].shift(1)
    df['lag_7'] = df.groupby(['item_id', 'store_id'], observed=False)['sales'].shift(7)
    df['lag_30'] = df.groupby(['item_id', 'store_id'], observed=False)['sales'].shift(30)

    # Rolling mean features
    df['rolling_mean_7'] = df.groupby(['item_id', 'store_id'], observed=False)['sales'].rolling(7).mean().reset_index(level=[0, 1], drop=True)
    df['rolling_mean_30'] = df.groupby(['item_id', 'store_id'], observed=False)['sales'].rolling(30).mean().reset_index(level=[0, 1], drop=True)

    # Fill NaN values with 0
    df.fillna(0, inplace=True)
    return df

# Create a dummy meta DataFrame to specify the output structure (needed for Dask)
meta = pd.DataFrame({
    'id': pd.Series([], dtype='int64'),
    'item_id': pd.Series([], dtype='category'),
    'dept_id': pd.Series([], dtype='category'),
    'cat_id': pd.Series([], dtype='category'),
    'store_id': pd.Series([], dtype='category'),
    'state_id': pd.Series([], dtype='category'),
    'date': pd.Series([], dtype='datetime64[ns]'),
    'sales': pd.Series([], dtype='float64'),
    'day': pd.Series([], dtype='int64'),
    'month': pd.Series([], dtype='int64'),
    'year': pd.Series([], dtype='int64'),
    'day_of_week': pd.Series([], dtype='int64'),
    'lag_1': pd.Series([], dtype='float64'),
    'lag_7': pd.Series([], dtype='float64'),
    'lag_30': pd.Series([], dtype='float64'),
    'rolling_mean_7': pd.Series([], dtype='float64'),
    'rolling_mean_30': pd.Series([], dtype='float64')
})

# Apply the function to the Dask DataFrame with the meta argument
sales_melted = sales_melted.map_partitions(create_time_series_features_dask, meta=meta)

# Convert Dask DataFrame to Pandas DataFrame (after feature engineering)
sales_melted_computed = sales_melted.compute()

# Label Encoding: Update the categories before applying label encoding
encoder = LabelEncoder()

# Ensure categories are updated before label encoding
sales_melted_computed['item_id'] = sales_melted_computed['item_id'].cat.set_categories(sales_melted_computed['item_id'].unique())
sales_melted_computed['store_id'] = sales_melted_computed['store_id'].cat.set_categories(sales_melted_computed['store_id'].unique())

sales_melted_computed['item_id'] = encoder.fit_transform(sales_melted_computed['item_id'])
sales_melted_computed['store_id'] = encoder.fit_transform(sales_melted_computed['store_id'])

# Split the data into training and test sets
train = sales_melted_computed[sales_melted_computed['date'] < '2016-01-01']
test = sales_melted_computed[sales_melted_computed['date'] >= '2016-01-01']

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

# Initialize and train the XGBoost model
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=1000, learning_rate=0.05)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f'RMSE: {rmse}')
print(f'MAE: {mae}')
